# Évaluer un prompt de classification

Un prompt se met au point comme un modèle de *machine learning* : on le mesure sur un **jeu de test** dont on connaît les bonnes réponses, plutôt que de le juger sur deux ou trois essais.

Ici, un LLM trie les tickets du support informatique en cinq catégories. On compare plusieurs versions du prompt sur 40 tickets étiquetés à la main.

Le modèle : [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), un petit modèle ouvert. Un GPU est recommandé (`Exécution > Modifier le type d'exécution`). Sans GPU, le notebook charge la version 0,6B : plus légère, mais nettement moins précise.

In [1]:
import warnings

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "1.7B" if device == "cuda" else "0.6B"  # 0.6B : assez léger pour un CPU

if device == "cpu":
    warnings.warn("Aucun GPU détecté : le notebook charge le modèle 0,6B, nettement moins précis. "
                  "Pour le modèle 1,7B, choisissez un GPU (Exécution > Modifier le type d'exécution), puis réexécutez le notebook.")

LLM = f"Qwen/Qwen3-{MODEL_SIZE}"
DTYPE = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LLM, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map=device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## Le jeu de test

40 tickets, 8 par catégorie, avec la catégorie qu'aurait choisie le support. Certains sont ambigus exprès : un e-mail d'hameçonnage parle de messagerie, mais relève de la sécurité ; un mot de passe VPN parle de réseau, mais relève de l'accès.

In [2]:
CATEGORIES = ["accès", "réseau", "messagerie", "poste", "sécurité"]

TICKETS = [
    ("Mon compte est verrouillé après plusieurs erreurs de mot de passe.", "accès"),
    ("Je n'ai pas les droits pour ouvrir le dossier partagé Finance.", "accès"),
    ("Mon mot de passe VPN a expiré, comment le changer ?", "accès"),
    ("Nouvel arrivant lundi : merci de créer son compte et sa boîte mail.", "accès"),
    ("Je ne reçois plus les codes de double authentification sur mon téléphone.", "accès"),
    ("Peut-on me donner accès en écriture au wiki de l'équipe ?", "accès"),
    ("J'ai oublié mon mot de passe Windows.", "accès"),
    ("Outlook me redemande mon mot de passe à chaque démarrage.", "accès"),
    ("Le VPN se déconnecte toutes les dix minutes.", "réseau"),
    ("Pas de wifi dans la salle de réunion B.", "réseau"),
    ("Internet est très lent depuis ce matin sur tout l'étage.", "réseau"),
    ("Impossible de joindre le serveur de fichiers depuis l'agence de Lyon.", "réseau"),
    ("Le VPN refuse de se connecter : erreur « passerelle injoignable ».", "réseau"),
    ("Les visioconférences coupent sans arrêt depuis le site de Nantes.", "réseau"),
    ("La prise réseau de mon bureau ne fonctionne plus.", "réseau"),
    ("Le site intranet ne répond plus, pour tout le monde.", "réseau"),
    ("Je ne reçois plus d'e-mails depuis hier soir.", "messagerie"),
    ("Ma boîte mail est pleine, je ne peux plus envoyer de messages.", "messagerie"),
    ("Comment partager mon calendrier avec mon assistante ?", "messagerie"),
    ("Un client me dit que mes e-mails arrivent dans ses indésirables.", "messagerie"),
    ("Je voudrais une liste de diffusion pour l'équipe projet.", "messagerie"),
    ("Une pièce jointe de 30 Mo est refusée à l'envoi.", "messagerie"),
    ("Mes réponses automatiques d'absence ne partent pas.", "messagerie"),
    ("Les invitations aux réunions arrivent avec une heure de décalage.", "messagerie"),
    ("Mon écran reste noir au démarrage.", "poste"),
    ("L'imprimante du 3e étage imprime des pages blanches.", "poste"),
    ("J'aurais besoin d'Excel sur mon nouveau portable.", "poste"),
    ("Mon ordinateur est très lent depuis la dernière mise à jour.", "poste"),
    ("Plusieurs touches du clavier de mon portable ne répondent plus.", "poste"),
    ("La batterie de mon portable ne tient plus qu'une heure.", "poste"),
    ("Le logiciel de comptabilité plante à l'ouverture.", "poste"),
    ("Mon casque n'est pas reconnu par l'ordinateur.", "poste"),
    ("Un e-mail de la « DSI » me demande de confirmer mon mot de passe via un lien.", "sécurité"),
    ("J'ai cliqué sur un lien dans un e-mail et une page m'a demandé mes identifiants.", "sécurité"),
    ("L'antivirus signale un fichier infecté sur ma clé USB.", "sécurité"),
    ("J'ai perdu mon ordinateur portable professionnel dans le train.", "sécurité"),
    ("Des fenêtres publicitaires s'ouvrent toutes seules sur mon poste.", "sécurité"),
    ("Quelqu'un s'est connecté à mon compte depuis l'étranger cette nuit.", "sécurité"),
    ("Un fournisseur demande par e-mail de changer son RIB avant le prochain virement.", "sécurité"),
    ("J'ai envoyé par erreur un fichier client à une adresse externe.", "sécurité"),
]
test_set = pd.DataFrame(TICKETS, columns=["ticket", "attendu"])
test_set.attendu.value_counts()

,count
attendu,
accès,8
réseau,8
messagerie,8
poste,8
sécurité,8


## Mesurer un prompt

La fonction `evaluer` envoie chaque ticket au modèle avec le prompt (`{ticket}` y est remplacé par le texte du ticket), lit la première catégorie citée dans la réponse, et calcule la proportion de bonnes réponses. Une réponse qui ne cite aucune catégorie compte comme une erreur, notée `?`.

In [3]:
def ask(prompts: list[str], batch_size: int = 8) -> list[str]:
    answers = []
    for start in range(0, len(prompts), batch_size):
        messages = [[{"role": "user", "content": p}] for p in prompts[start:start + batch_size]]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                               padding=True, return_tensors="pt", return_dict=True).to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=30, do_sample=False, temperature=None, top_p=None,
                                    top_k=None, pad_token_id=tokenizer.eos_token_id)
        answers += tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return answers


def category(answer: str) -> str:
    answer = answer.lower()
    found = [(answer.find(c), c) for c in CATEGORIES if c in answer]
    return min(found)[1] if found else "?"


def evaluer(prompt: str) -> pd.DataFrame:
    answers = ask([prompt.replace("{ticket}", t) for t in test_set.ticket])
    results = test_set.assign(réponse=answers, prédit=[category(a) for a in answers])
    accuracy = (results.prédit == results.attendu).mean()
    print(f"Exactitude : {accuracy:.0%} ({(results.prédit == results.attendu).sum()}/{len(results)})")
    return results


def erreurs(results: pd.DataFrame) -> pd.DataFrame:
    return results[results.prédit != results.attendu][["ticket", "attendu", "prédit"]]

## Version 1 : le prompt naïf

In [4]:
PROMPT_V1 = """Classe ce ticket dans une catégorie : accès, réseau, messagerie, poste ou sécurité.

Ticket : {ticket}"""

v1 = evaluer(PROMPT_V1)
v1.réponse.head()

Exactitude : 42% (17/40)


,réponse
0,Le ticket doit être classé dans la catégorie *...
1,Ce ticket doit être classé dans la catégorie *...
2,Le ticket se classe dans la catégorie **accès**.
3,Le ticket doit être classé dans la catégorie *...
4,Ce ticket doit être classé dans la catégorie *...


Le modèle répond par des phrases de longueur variable : facile à lire pour un humain, moins pour un programme. Quelles erreurs fait-il ?

In [5]:
erreurs(v1)

,ticket,attendu,prédit
3,Nouvel arrivant lundi : merci de créer son com...,accès,messagerie
7,Outlook me redemande mon mot de passe à chaque...,accès,messagerie
11,Impossible de joindre le serveur de fichiers d...,réseau,accès
18,Comment partager mon calendrier avec mon assis...,messagerie,accès
20,Je voudrais une liste de diffusion pour l'équi...,messagerie,accès
21,Une pièce jointe de 30 Mo est refusée à l'envoi.,messagerie,accès
22,Mes réponses automatiques d'absence ne partent...,messagerie,réseau
23,Les invitations aux réunions arrivent avec une...,messagerie,réseau
24,Mon écran reste noir au démarrage.,poste,accès
25,L'imprimante du 3e étage imprime des pages bla...,poste,accès


## Version 2 : définir les catégories et le format

Le modèle ne connaît pas nos conventions : où ranger un mot de passe VPN, ou un e-mail d'hameçonnage ? On définit chaque catégorie, avec les règles de priorité, et on impose le format de la réponse.

In [6]:
PROMPT_V2 = """Tu tries les tickets du support informatique. Catégories :
- accès : comptes, mots de passe (y compris celui du VPN ou de la messagerie), droits, double authentification
- réseau : VPN, wifi, connexion internet, lenteur ou coupure du réseau, sites internes injoignables
- messagerie : e-mails, calendrier, listes de diffusion, pièces jointes
- poste : matériel et logiciels du poste de travail, imprimantes
- sécurité : e-mail suspect ou frauduleux, virus, vol ou perte de matériel, connexion suspecte, fuite de données

Règle : tout ce qui touche à une menace ou à un incident de sécurité relève de « sécurité », même si cela passe par un e-mail ou un compte.

Réponds uniquement par le nom de la catégorie, en minuscules.

Ticket : {ticket}"""

v2 = evaluer(PROMPT_V2)
erreurs(v2)

Exactitude : 65% (26/40)


,ticket,attendu,prédit
16,Je ne reçois plus d'e-mails depuis hier soir.,messagerie,réseau
17,"Ma boîte mail est pleine, je ne peux plus envo...",messagerie,réseau
18,Comment partager mon calendrier avec mon assis...,messagerie,sécurité
19,Un client me dit que mes e-mails arrivent dans...,messagerie,sécurité
20,Je voudrais une liste de diffusion pour l'équi...,messagerie,poste
21,Une pièce jointe de 30 Mo est refusée à l'envoi.,messagerie,accès
22,Mes réponses automatiques d'absence ne partent...,messagerie,réseau
23,Les invitations aux réunions arrivent avec une...,messagerie,réseau
24,Mon écran reste noir au démarrage.,poste,?
25,L'imprimante du 3e étage imprime des pages bla...,poste,?


## Version 3 : ajouter des exemples

On ajoute quelques exemples résolus, choisis parmi les cas difficiles, mais **absents du jeu de test** : sinon, on mesurerait la mémoire du modèle, pas sa capacité à généraliser.

In [7]:
EXAMPLES = """Exemples :
Ticket : Mon accès au VPN est bloqué après trois mauvais mots de passe. → accès
Ticket : Le partage d'écran se fige pendant les réunions à distance avec l'agence. → réseau
Ticket : Je ne retrouve plus mes anciens e-mails archivés. → messagerie
Ticket : Ma souris sans fil ne fonctionne plus. → poste
Ticket : Un SMS au nom de la banque de l'entreprise me demande mes identifiants. → sécurité

"""

PROMPT_V3 = PROMPT_V2.replace("Ticket : {ticket}", EXAMPLES + "Ticket : {ticket}")

v3 = evaluer(PROMPT_V3)
erreurs(v3)

Exactitude : 72% (29/40)


,ticket,attendu,prédit
3,Nouvel arrivant lundi : merci de créer son com...,accès,sécurité
4,Je ne reçois plus les codes de double authenti...,accès,sécurité
7,Outlook me redemande mon mot de passe à chaque...,accès,sécurité
16,Je ne reçois plus d'e-mails depuis hier soir.,messagerie,sécurité
18,Comment partager mon calendrier avec mon assis...,messagerie,poste
19,Un client me dit que mes e-mails arrivent dans...,messagerie,sécurité
22,Mes réponses automatiques d'absence ne partent...,messagerie,poste
23,Les invitations aux réunions arrivent avec une...,messagerie,réseau
27,Mon ordinateur est très lent depuis la dernièr...,poste,réseau
35,J'ai perdu mon ordinateur portable professionn...,sécurité,poste


## Comparer les versions

Les bonnes réponses par catégorie, pour chaque version :

In [8]:
pd.DataFrame({
    version: (results.prédit == results.attendu).groupby(results.attendu).sum()
    for version, results in {"v1": v1, "v2": v2, "v3": v3}.items()
})

,v1,v2,v3
attendu,,,
accès,6,8,5
messagerie,3,0,3
poste,0,3,7
réseau,7,8,8
sécurité,1,7,6


Chaque version corrige des erreurs et peut en créer d'autres : avec ses définitions, la v2 fait passer la sécurité de 1 à 7 bonnes réponses sur 8, mais la messagerie tombe à 0 : le modèle range le plus souvent les e-mails qui n'arrivent plus dans « réseau ». Les exemples de la v3 redressent le poste et la messagerie, mais trois problèmes de compte partent à leur tour dans « sécurité ». De la v2 à la v3, le total ne gagne que 3 tickets, et cache des pertes : d'où l'intérêt de mesurer catégorie par catégorie, pas seulement au total.

La matrice de confusion de la meilleure version : en ligne la catégorie attendue, en colonne la catégorie prédite.

In [9]:
pd.crosstab(v3.attendu, v3.prédit)

prédit,accès,messagerie,poste,réseau,sécurité
attendu,,,,,
accès,5,0,0,0,3
messagerie,0,3,2,1,2
poste,0,0,7,1,0
réseau,0,0,0,8,0
sécurité,0,0,2,0,6


Avec 40 tickets, un ticket vaut 2,5 points : un écart de 5 points entre deux versions peut tenir au hasard. Un vrai jeu de test compte plutôt une centaine de cas, tirés des tickets réels.

## À vous

Écrivez votre propre version du prompt, avec `{ticket}` à l'endroit du ticket, et mesurez-la. Par exemple, cherchez à redresser la catégorie la plus faible de la v3.

In [10]:
MON_PROMPT = """Classe ce ticket du support informatique : accès, réseau, messagerie, poste ou sécurité.
Réponds par un seul mot.

Ticket : {ticket}"""

mien = evaluer(MON_PROMPT)
erreurs(mien)

Exactitude : 35% (14/40)


,ticket,attendu,prédit
1,Je n'ai pas les droits pour ouvrir le dossier ...,accès,réseau
2,"Mon mot de passe VPN a expiré, comment le chan...",accès,réseau
6,J'ai oublié mon mot de passe Windows.,accès,réseau
7,Outlook me redemande mon mot de passe à chaque...,accès,messagerie
11,Impossible de joindre le serveur de fichiers d...,réseau,?
18,Comment partager mon calendrier avec mon assis...,messagerie,?
20,Je voudrais une liste de diffusion pour l'équi...,messagerie,réseau
21,Une pièce jointe de 30 Mo est refusée à l'envoi.,messagerie,réseau
22,Mes réponses automatiques d'absence ne partent...,messagerie,réseau
23,Les invitations aux réunions arrivent avec une...,messagerie,réseau


## À retenir

- Un prompt se juge sur un jeu de test étiqueté, pas sur quelques essais
- Définir les catégories et nos conventions, imposer le format, donner des exemples : chaque ajout se mesure
- Les exemples du prompt ne doivent pas venir du jeu de test
- Ce jeu de test se rejoue à chaque changement de prompt ou de modèle